In [ ]:
# ACE-Step 1.5 편집 화면(Gradio) 자동 설치 · 실행 — Kaggle T4 (2026-10-31)
# 이 셀 하나만 실행하면 된다. 마지막에 나오는 https://xxxx.gradio.live 주소를 Claude 에게 주면 된다.
# 설정: Internet 켜기, GPU T4 x2(또는 T4)
MODEL = "acestep-v15-turbo"   # T4(16GB) 기본. 편곡에서 원곡 선율을 지키는 "acestep-v15-xl-base" 는 float32 로 16GB 를 넘어 24GB 이상 GPU 필요

import os, subprocess
os.chdir("/kaggle/working")
def sh(c):
    print("$", c, flush=True); subprocess.run(c, shell=True, check=True)

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
if not os.path.exists("ACE-Step-1.5"):
    sh("git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git")
os.chdir("ACE-Step-1.5")
sh("pip install -q uv")
# 함정 1: T4 는 bf16 이 없어 float16 을 쓰는데 cover · lego 에서 NaN(무음 · 잡음) → float32 로 (10/25 확인)
sh("sed -i 's/self.dtype = torch.float16/self.dtype = torch.float32/' acestep/core/generation/handler/init_service_orchestrator.py || true")
sh("uv sync 2>&1 | tail -3")
# 함정 2: 노트북의 MPLBACKEND(matplotlib_inline) 가 넘어가면 하위 프로세스가 죽는다
os.environ["MPLBACKEND"] = "Agg"
# 시작할 때 모델까지 올려 둔다(화면에서 Initialize 안 눌러도 됨). INT8 양자화는 끈다 — T4 에서 NaN 잠재값이 난다(AIQUEST 노트북도 같은 이유로 끔)
sh(f"uv run acestep --server-name 0.0.0.0 --share --init_service true --config_path {MODEL} "
   f"--init_llm false --quantization none --offload_to_cpu true --offload_dit_to_cpu true")
